# JsonOutputParser

`JsonOutputParser`는 LLM의 답변을 **JSON 형태로 변환**해서 받을 때 사용하는 출력 파서입니다.

특히 원하는 데이터 구조를 미리 정해두면, LLM의 답변을 일정한 형식으로 받을 수 있습니다.

## 핵심

- LLM 결과를 JSON 형태로 반환
- 원하는 키와 값의 구조를 지정 가능
- Pydantic과 함께 사용하면 출력 구조를 더 명확하게 제한할 수 있음

## JSON이란?

JSON은 데이터를 `key : value` 형태로 표현하는 데이터 형식입니다.

- `{ }` : 객체
- `[ ]` : 배열
- `key : value` : 하나의 데이터 항목

예시:

```json
{
  "name": "John Doe",
  "age": 30,
  "is_student": false,
  "skills": ["Java", "Python", "JavaScript"],
  "address": {
    "street": "123 Main St",
    "city": "Anytown"
  }
}

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [1]:
# LangSmith 추적 설정
# !pip install langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름 설정
logging.langsmith("CH03-OutputParser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [3]:
# 사용할 OpenAI 모델 생성
model = ChatOpenAI(temperature=0, model_name="gpt-4.1-mini")

원하는 JSON 출력 구조를 Pydantic으로 정의합니다.

In [5]:
# 원하는 출력 데이터 구조 정의
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: str = Field(description="해시태그 형식의 키워드(2개 이상)")

`JsonOutputParser`에 `Topic` 구조를 전달하고, LLM이 해당 형식으로 답하도록 지시합니다.

In [6]:
# 질문 작성
question = "지구 온난화의 심각성 대해 알려주세요."

# Topic 구조를 기준으로 JSON 출력 파서 생성
parser = JsonOutputParser(pydantic_object=Topic)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [7]:
# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# Prompt → Model → JsonOutputParser 연결
chain = prompt | model | parser

# 체인 실행
answer = chain.invoke({"question": question})

In [8]:
# 반환된 데이터 타입 확인
type(answer)

dict

In [9]:
# JSON 결과 확인
answer

{'description': '지구 온난화는 지구 평균 기온이 상승하여 기후 변화, 해수면 상승, 생태계 파괴 등 심각한 환경 문제를 초래합니다.',
 'hashtags': '#지구온난화 #기후변화 #환경문제 #지구보호'}

## Pydantic 없이 JsonOutputParser 사용하기

`JsonOutputParser`는 Pydantic 없이도 사용할 수 있습니다.

다만 Pydantic을 사용하지 않으면 `description`, `hashtags`처럼 정확한 스키마를 강하게 지정하지 않고, 질문과 프롬프트를 통해 원하는 JSON 구조를 설명해야 합니다.

In [11]:
# 질문 작성
question = "지구 온난화에 대해 알려주세요. 온난화에 대한 설명은 `description`에, 관련 키워드는 `hashtags`에 담아주세요."

# Pydantic 없이 JSON 출력 파서 생성
parser = JsonOutputParser()

# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

# 출력 형식 지시사항을 프롬프트에 추가
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# Prompt → Model → Parser 연결
chain = prompt | model | parser

# 체인 실행
response = chain.invoke({"question": question})

# 결과 확인
print(response)

{'description': '지구 온난화는 대기 중 온실가스 농도의 증가로 인해 지구 평균 기온이 상승하는 현상입니다. 이는 극지방의 빙하 감소, 해수면 상승, 이상 기후 현상 등 다양한 환경 변화를 초래합니다.', 'hashtags': ['#지구온난화', '#기후변화', '#온실가스', '#환경보호', '#지구환경']}
